In [ ]:
import torch
import math
import matplotlib.pyplot as plt

from src.knots import *
from src.extensions import *
from src.full_models import HyperbolicMinimalSurfacePINN
from src.plotting import plot_error, plot_knot, montecarlo_error, plot_mu_heatmap_log
from src.geometry import minimal_in_H4_PDE_flat_new
from src.samplers import MixSampler
from src.training import train_PINN_Adam, refine_PINN_lbfgs
from src.double_point_analysis import find_candidate_double_points, refine_double_points_newton
from src.interior_models import MLP

from mpl_toolkits.mplot3d import Axes3D
%matplotlib widget

In [ ]:
model_names = []
# model_names.append('trained_models/models/paper/KNOT_figure8_torus_KNOT_PAR_A0.45_R1_mirrorFalse_PERTURBED.pt')
model_names.append('trained_models/models/paper/KNOT_figure8_torus_KNOT_PAR_A0.45_R1_mirrorTrue_PERTURBED.pt')
model_names.append('trained_models/models/paper/KNOT_square_KNOT_PAR_R0.6_PERTURBED.pt')
model_names.append('trained_models/models/paper/KNOT_stevedore_KNOT_PAR_R0.95_mirrorTrue_PERTURBED.pt')
model_names.append('trained_models/models/paper/KNOT_stevedore_KNOT_PAR_R1.6_mirrorFalse_PERTURBED.pt')
model_names.append('trained_models/models/paper/KNOT_three_twist_KNOT_PAR_R0.6_mirrorFalse_PERTURBED.pt')
model_names.append('trained_models/models/paper/KNOT_three_twist_KNOT_PAR_R0.6_mirrorTrue_PERTURBED.pt')
model_names.append('trained_models/models/paper/KNOT_torus_KNOT_PAR_R1_p3_q2_r0.5_PERTURBED.pt')
model_names.append('trained_models/models/paper/KNOT_torus_KNOT_PAR_R1_p4_q3_r0.5_PERTURBED.pt')
model_names.append('trained_models/models/paper/KNOT_torus_KNOT_PAR_R1_p4_q3_r0.5.pt')
model_names.append('trained_models/models/paper/KNOT_torus_KNOT_PAR_R1_p5_q2_r0.5_PERTURBED.pt')
model_names.append('trained_models/models/paper/KNOT_torus_KNOT_PAR_R1_p5_q3_r0.5_PERTURBED.pt')
model_names.append('trained_models/models/paper/KNOT_torus_KNOT_PAR_R2_p5_q3_r1_PERTURBED.pt')
model_names.append('trained_models/models/paper/KNOT_unknot_KNOT_PAR_R1_PERTURBED.pt')

In [ ]:
for model_name in model_names:
    model = HyperbolicMinimalSurfacePINN.load(model_name)
    print(model.name)
    
    sampler = MixSampler(mix=1, bias=0.5)
    n_points = 2**14

    xy = sampler(n_points)
    t = minimal_in_H4_PDE_flat_new(model)(xy)
    print('Max pointwise norm of the tension field:', torch.max((t**2).sum(dim=-1)**0.5).detach().item())
    print('MSE:', (t**2).sum(dim=-1).mean().detach().item())

    montecarlo_error(
        minimal_in_H4_PDE_flat_new(model),
        num_samples=100,
        size_samples=n_points,
        mix=1,
        bias=0.5,
        
        figsize=(5,3),
        bins=100,
        title=None,
        xlabel = 'Loss',
    )

    xy = sampler(n_points)
    _ = refine_PINN_lbfgs(
        model,
        xy_grid=xy,              
        lr=1.0,
        max_iter=10000,
        log_every=10,
        tolerance_grad=1e-15,
        tolerance_change=1e-16,
    )

    xy = sampler(n_points)
    t = minimal_in_H4_PDE_flat_new(model)(xy)
    print('Max pointwise norm of the tension field:', torch.max((t**2).sum(dim=-1)**0.5).detach().item())
    print('MSE:', (t**2).sum(dim=-1).mean().detach().item())

    montecarlo_error(
        minimal_in_H4_PDE_flat_new(model),
        num_samples=100,
        size_samples=n_points,
        mix=1,
        bias=0.5,
        
        figsize=(5,3),
        bins=100,
        title=None,
        xlabel = 'Loss',
    )

    model.save(filepath='trained_models/models/paper/refined/' + model.name)

## Self-proximity map

In [ ]:
plot_mu_heatmap_log(
    # --- Main settings
    u_callable = model,
    epsilon = 0.3,
    cmap='viridis_r',
    grid_resolution=300,
    figsize=(6,5),
    title = None,

    # --- Boundary Settings ---
    boundary_color='black', 
    boundary_linewidth=2.6,

    # --- Colorbar Settings ---
    show_colorbar=True,
    colorbar_label=None,
)

In [ ]:
candidates = find_candidate_double_points(
    model,
    grid_resolution=200,
    epsilon=0.2,
    tau=0.03,
    dtype=model.kwargs['dtype'])

candidates

In [ ]:
refined_pairs, jacobians, final_distances = refine_double_points_newton(
    model,
    candidates,
    tol=1e-14,
    max_iter=200,
    dedup_tol=1e-11,
    dtype=model.kwargs['dtype'],
)

In [ ]:
refined_pairs

In [ ]:
jacobians

In [ ]:
final_distances

In [ ]:
# Bundle A - color
plot_mu_heatmap_log(
    # --- Main settings
    u_callable = model,
    epsilon = 0.3,
    cmap='viridis_r',
    grid_resolution=300,
    figsize=(6,5),
    title = 'None',

    # # --- Candidate Settings ---
    # candidates=candidates,
    # candidate_color="#E85D04",
    # alpha = 0.8,

    # --- Refined pairs Settings ---
    refined_pairs=refined_pairs,
    jacobians=jacobians,
    refined_pos_color='#ffc000',
    refined_neg_color="#a40e26",

    # --- Legend Settings ---
    # legend_title = 'Double points',
    legend_label_candidates='Candidate double points',
    legend_label_refined_pos=r'Positive double points',
    legend_label_refined_neg=r'Negative double points',
    # legend_bbox_to_anchor=None,
    # legend_loc='upper right',
    legend_frameon=True,

    # --- Boundary Settings ---
    boundary_color='black', 
    boundary_linewidth=2.6,

    # --- Colorbar Settings ---
    show_colorbar=True,
    colorbar_label=None,
)

In [ ]:
# Bundle A - greyscale
plot_mu_heatmap_log(
    # --- Main settings
    u_callable = model,
    epsilon = 0.3,
    cmap='Greys',
    grid_resolution=100,
    title = None,

    # # --- Candidate Settings ---
    # candidates=candidates,
    # candidate_color="#5B5A5A",
    # alpha = 0.5,
    # legend_label_candidates='Candidate pairs',

    # --- Refined pairs Settings ---
    refined_pairs=refined_pairs,
    jacobians=jacobians,
    refined_pos_color="#000000",
    refined_neg_color="#888888",

    # --- Legend Settings ---
    legend_title = 'Double points',
    legend_label_candidates='Candidate',
    legend_label_refined_pos=r'Positive',
    legend_label_refined_neg=r'Negative',
    legend_bbox_to_anchor=None,
    legend_loc='upper right',
    legend_frameon=True,

    # --- Boundary Settings ---
    boundary_color='black', 
    boundary_linewidth=2.5,

    # --- Other Settings ---
    colorbar_label=None,
)